# Piloto — Da rede lexical à rede contextual em Transformers

Este notebook implementa um piloto pequeno do projeto de Redes Complexas.

A ideia é acompanhar **as mesmas ocorrências de tokens** desde o embedding lexical de entrada até algumas camadas internas de um Transformer e representar cada estágio como uma rede k-NN.

## Objetivos do piloto

1. Tokenizar um texto curto em português (~100–150 tokens).
2. Tratar cada ocorrência de token como um vértice distinto.
3. Extrair:
   - embedding lexical de entrada;
   - hidden state de uma camada inicial;
   - hidden state de uma camada intermediária;
   - hidden state de uma camada final.
4. Construir uma rede k-NN para cada estágio.
5. Comparar:
   - métricas estruturais;
   - reorganização de vizinhanças por Jaccard;
   - dominância lexical;
   - comunidades;
   - ocorrências repetidas em contextos distintos.

> Este notebook é um **piloto metodológico**. O objetivo não é produzir conclusões científicas robustas, mas validar o pipeline antes de escalar para aproximadamente 15 mil ocorrências.

## 1. Configuração e dependências

In [ ]:
# Se necessário, instale as dependências:
# %pip install -q torch transformers pandas numpy scikit-learn networkx matplotlib python-louvain

import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import torch

from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    import community as community_louvain  # pacote python-louvain
    HAS_LOUVAIN = True
except Exception:
    HAS_LOUVAIN = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
K = 5
MAX_CONTEXT_CHARS = 45

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("DEVICE:", DEVICE)
print("MODEL_ID:", MODEL_ID)
print("k:", K)

: 

### Observação sobre o modelo

O piloto usa `Qwen/Qwen2.5-0.5B-Instruct` por ser pequeno e simples de executar em uma máquina comum.

No experimento final, o `MODEL_ID` pode ser substituído por outro Transformer autoregressivo aberto, mantendo o restante do pipeline praticamente igual.

## 2. Texto piloto

In [ ]:
TEXT = """
O banco aprovou o empréstimo depois de analisar o crédito do cliente.
Mais tarde, o mesmo cliente sentou em um banco de madeira na praça e observou as pessoas.
Uma rede de computadores conectava os laboratórios da universidade, enquanto uma rede de pesca secava perto do rio.
Em outra sala, pesquisadores estudavam uma rede social e discutiam como as pessoas formam comunidades.
A manga madura caiu da árvore, mas a manga da camisa continuava dobrada sobre a cadeira.
A capital recebeu novos investimentos, enquanto o empresário explicou que precisava ampliar o capital da empresa.
No parque, o banco estava molhado pela chuva. No centro financeiro, outro banco anunciou novas taxas de juros.
A rede do laboratório caiu novamente, e a equipe tentou restabelecer a conexão antes da reunião.
"""

print(TEXT.strip())

## 3. Carregamento do tokenizer e inspeção da tokenização

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

encoded = tokenizer(
    TEXT,
    return_tensors="pt",
    add_special_tokens=False
)

input_ids = encoded["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids.tolist())

print("Número de ocorrências:", len(input_ids))
print("Número de token_ids distintos:", len(set(input_ids.tolist())))
print("Primeiros 30 tokens:")
print(tokens[:30])

In [ ]:
def decode_token_piece(token_id):
    return tokenizer.decode([int(token_id)], clean_up_tokenization_spaces=False)

token_rows = []
for pos, token_id in enumerate(input_ids.tolist()):
    token_rows.append({
        "position": pos,
        "token_id": int(token_id),
        "token": tokens[pos],
        "decoded_piece": decode_token_piece(token_id),
    })

tokens_df = pd.DataFrame(token_rows)
tokens_df.head(30)

### Contagem de tokens repetidos

Aqui usamos `token_id`, não palavras reconstruídas. Isso é importante porque o tokenizer pode dividir uma palavra em múltiplos subtokens.

In [ ]:
freq = Counter(input_ids.tolist())

tokens_df["frequency"] = tokens_df["token_id"].map(freq)

repeated_types_df = (
    tokens_df[tokens_df["frequency"] > 1]
    .drop_duplicates("token_id")
    .sort_values(["frequency", "token_id"], ascending=[False, True])
)

print("Tipos/token_ids repetidos:", len(repeated_types_df))
display(repeated_types_df.head(30))

## 4. Contexto local de cada ocorrência

In [ ]:
# Contexto aproximado no nível de tokens, suficiente para inspeção manual.
def local_token_context(position, radius=6):
    start = max(0, position - radius)
    end = min(len(input_ids), position + radius + 1)
    ids = input_ids[start:end].tolist()
    return tokenizer.decode(ids, skip_special_tokens=True)

tokens_df["context"] = [
    local_token_context(i, radius=6)
    for i in range(len(tokens_df))
]

display(tokens_df.head(20))

## 5. Carregamento do Transformer

In [ ]:
dtype = torch.float16 if DEVICE == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
)

model = model.to(DEVICE)
model.eval()

print("Modelo carregado.")
print("Número de parâmetros:", f"{sum(p.numel() for p in model.parameters()):,}")

## 6. Forward pass e extração dos hidden states

In [ ]:
inputs = {
    k: v.to(DEVICE)
    for k, v in encoded.items()
}

with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

hidden_states = outputs.hidden_states

print("Quantidade de tensores em hidden_states:", len(hidden_states))
print("Shape de hidden_states[0]:", hidden_states[0].shape)
print("Shape de hidden_states[-1]:", hidden_states[-1].shape)

### Convenção dos índices

Em modelos Hugging Face, `hidden_states[0]` normalmente corresponde à representação após a etapa de embedding e antes do primeiro bloco Transformer.

Os índices seguintes correspondem às saídas sucessivas dos blocos.

O notebook registra explicitamente quais índices foram utilizados.

In [ ]:
num_hidden_tensors = len(hidden_states)
num_transformer_blocks = num_hidden_tensors - 1

initial_idx = 1 if num_transformer_blocks >= 1 else 0
middle_idx = max(1, num_transformer_blocks // 2)
final_idx = num_transformer_blocks

SELECTED_STAGES = {
    "lexical": 0,
    "initial": initial_idx,
    "middle": middle_idx,
    "final": final_idx,
}

print("Blocos Transformer:", num_transformer_blocks)
print("Índices selecionados:", SELECTED_STAGES)

## 7. Extração do embedding lexical

In [ ]:
# A embedding matrix lexical é obtida diretamente da camada de entrada do modelo.
embedding_layer = model.get_input_embeddings()

with torch.no_grad():
    lexical_embeddings = embedding_layer(inputs["input_ids"])[0]

print("Shape lexical:", lexical_embeddings.shape)

In [ ]:
representations = {
    "lexical": lexical_embeddings.detach().float().cpu().numpy(),
    "initial": hidden_states[initial_idx][0].detach().float().cpu().numpy(),
    "middle": hidden_states[middle_idx][0].detach().float().cpu().numpy(),
    "final": hidden_states[final_idx][0].detach().float().cpu().numpy(),
}

for stage, arr in representations.items():
    print(stage, arr.shape)

## 8. Sanity checks das representações

In [ ]:
n_vertices = len(tokens_df)

assert all(arr.shape[0] == n_vertices for arr in representations.values())

print("Mesmo número de vértices em todos os estágios:", n_vertices)

In [ ]:
# Verifica se ocorrências do mesmo token_id possuem embeddings lexicais idênticos.
same_token_pairs_checked = 0
same_token_lexical_equal = 0

positions_by_token = defaultdict(list)
for pos, token_id in enumerate(input_ids.tolist()):
    positions_by_token[token_id].append(pos)

for token_id, positions in positions_by_token.items():
    if len(positions) >= 2:
        base = representations["lexical"][positions[0]]
        for p in positions[1:]:
            same_token_pairs_checked += 1
            if np.allclose(base, representations["lexical"][p], atol=1e-6):
                same_token_lexical_equal += 1

print("Pares repetidos verificados:", same_token_pairs_checked)
print("Pares lexicalmente idênticos:", same_token_lexical_equal)

assert same_token_pairs_checked == same_token_lexical_equal

In [ ]:
# Verifica se pelo menos algumas ocorrências repetidas deixam de ser idênticas contextualmente.
contextual_distances = []

for token_id, positions in positions_by_token.items():
    if len(positions) >= 2:
        p0 = positions[0]
        for p1 in positions[1:]:
            d = np.linalg.norm(
                representations["final"][p0] - representations["final"][p1]
            )
            contextual_distances.append((token_id, p0, p1, d))

contextual_distances_df = pd.DataFrame(
    contextual_distances,
    columns=["token_id", "position_a", "position_b", "final_l2_distance"]
).sort_values("final_l2_distance", ascending=False)

display(contextual_distances_df.head(20))

## 9. Similaridade cosseno

In [ ]:
similarity_matrices = {
    stage: cosine_similarity(vectors)
    for stage, vectors in representations.items()
}

for stage, sim in similarity_matrices.items():
    print(
        stage,
        "min =", float(sim.min()),
        "max =", float(sim.max()),
        "mean =", float(sim.mean())
    )

## 10. Construção das redes k-NN

Cada ocorrência é um vértice distinto.

Para cada vértice, conectamos seus `k` vizinhos mais próximos por similaridade cosseno.

### Regra de desempate

Quando duas similaridades são iguais, usamos a posição do vértice como critério secundário. Isso torna o resultado determinístico.

### Simetrização

O k-NN inicialmente é direcionado. O grafo final é não direcionado usando a **união** das relações:

- se `i` escolhe `j`, a aresta `{i,j}` existe;
- não é necessário que `j` também escolha `i`.

Self-loops são removidos.

In [ ]:
def deterministic_knn(similarity_matrix, k):
    n = similarity_matrix.shape[0]
    neighbors = {}

    for i in range(n):
        candidates = [
            (j, similarity_matrix[i, j])
            for j in range(n)
            if j != i
        ]

        # Maior similaridade primeiro; posição j como desempate.
        candidates.sort(key=lambda x: (-x[1], x[0]))
        neighbors[i] = [j for j, _ in candidates[:k]]

    return neighbors


def build_undirected_knn_graph(similarity_matrix, neighbors):
    G = nx.Graph()
    n = similarity_matrix.shape[0]
    G.add_nodes_from(range(n))

    for i, nbrs in neighbors.items():
        for j in nbrs:
            if i == j:
                continue
            weight = float(similarity_matrix[i, j])
            if G.has_edge(i, j):
                G[i][j]["weight"] = max(G[i][j]["weight"], weight)
            else:
                G.add_edge(i, j, weight=weight)

    return G

In [ ]:
neighbors_by_stage = {}
graphs = {}

for stage, sim in similarity_matrices.items():
    nbrs = deterministic_knn(sim, K)
    G = build_undirected_knn_graph(sim, nbrs)

    neighbors_by_stage[stage] = nbrs
    graphs[stage] = G

    assert G.number_of_nodes() == n_vertices
    assert nx.number_of_selfloops(G) == 0

    print(stage, "|V| =", G.number_of_nodes(), "|E| =", G.number_of_edges())

## 11. Métricas estruturais

In [ ]:
def graph_metrics(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()

    components = list(nx.connected_components(G))
    largest_nodes = max(components, key=len) if components else set()
    giant = G.subgraph(largest_nodes).copy()

    avg_degree = (
        sum(dict(G.degree()).values()) / n
        if n > 0 else np.nan
    )

    metrics = {
        "vertices": n,
        "edges": m,
        "avg_degree": avg_degree,
        "density": nx.density(G),
        "avg_clustering": nx.average_clustering(G),
        "components": len(components),
        "largest_component": len(largest_nodes),
        "avg_shortest_path_lcc": (
            nx.average_shortest_path_length(giant)
            if giant.number_of_nodes() > 1 else 0.0
        ),
        "diameter_lcc": (
            nx.diameter(giant)
            if giant.number_of_nodes() > 1 else 0
        ),
    }

    return metrics


metrics_df = pd.DataFrame({
    stage: graph_metrics(G)
    for stage, G in graphs.items()
}).T

metrics_df

## 12. Layout fixo para comparação visual

In [ ]:
# Para facilitar a comparação visual, calculamos o layout uma única vez
# usando uma união de todas as arestas observadas.
union_graph = nx.Graph()
union_graph.add_nodes_from(range(n_vertices))

for G in graphs.values():
    union_graph.add_edges_from(G.edges())

fixed_pos = nx.spring_layout(
    union_graph,
    seed=SEED,
    iterations=200
)

print("Layout fixo calculado.")

## 13. Visualização das redes

In [ ]:
def plot_graph_stage(stage, figsize=(10, 8), label_top_n=12):
    G = graphs[stage]

    plt.figure(figsize=figsize)

    degrees = dict(G.degree())
    node_sizes = [
        25 + 8 * degrees[n]
        for n in G.nodes()
    ]

    nx.draw_networkx_edges(
        G,
        fixed_pos,
        alpha=0.20,
        width=0.8
    )

    nx.draw_networkx_nodes(
        G,
        fixed_pos,
        node_size=node_sizes,
        alpha=0.80
    )

    # Rotula somente alguns nós de maior frequência lexical.
    label_candidates = (
        tokens_df
        .sort_values(["frequency", "position"], ascending=[False, True])
        .head(label_top_n)
    )

    labels = {
        int(row.position): f"{row.decoded_piece!r}\n{int(row.position)}"
        for _, row in label_candidates.iterrows()
    }

    nx.draw_networkx_labels(
        G,
        fixed_pos,
        labels=labels,
        font_size=7
    )

    plt.title(f"Rede k-NN — {stage}")
    plt.axis("off")
    plt.show()


for stage in ["lexical", "initial", "middle", "final"]:
    plot_graph_stage(stage)

## 14. Reorganização das vizinhanças por Jaccard

In [ ]:
def jaccard_neighbors(neighbors_a, neighbors_b):
    A = set(neighbors_a)
    B = set(neighbors_b)

    union = A | B
    if not union:
        return 1.0

    return len(A & B) / len(union)


jaccard_rows = []

comparisons = [
    ("lexical", "initial"),
    ("lexical", "middle"),
    ("lexical", "final"),
    ("initial", "middle"),
    ("middle", "final"),
]

for pos in range(n_vertices):
    row = {"position": pos}

    for a, b in comparisons:
        row[f"{a}_to_{b}"] = jaccard_neighbors(
            neighbors_by_stage[a][pos],
            neighbors_by_stage[b][pos]
        )

    jaccard_rows.append(row)

jaccard_df = pd.DataFrame(jaccard_rows)
jaccard_df.head()

In [ ]:
jaccard_summary = (
    jaccard_df
    .drop(columns=["position"])
    .agg(["mean", "median", "std", "min", "max"])
    .T
)

jaccard_summary

In [ ]:
summary_plot = (
    jaccard_df
    .drop(columns=["position"])
    .agg(["mean", "median"])
    .T
)

summary_plot.plot(
    kind="bar",
    figsize=(10, 5)
)
plt.ylabel("Jaccard")
plt.title("Persistência das vizinhanças entre estágios")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Ocorrências que mais mudaram do lexical para o final

In [ ]:
jaccard_inspection = (
    jaccard_df[["position", "lexical_to_final"]]
    .merge(
        tokens_df[["position", "decoded_piece", "token_id", "context"]],
        on="position"
    )
    .sort_values("lexical_to_final")
)

display(jaccard_inspection.head(20))

## 15. Dominância lexical

In [ ]:
def same_token_neighbor_fraction(position, neighbors, token_ids):
    own_id = int(token_ids[position])
    nbrs = neighbors[position]

    if not nbrs:
        return np.nan

    same = sum(
        int(token_ids[j]) == own_id
        for j in nbrs
    )

    return same / len(nbrs)


dominance_rows = []

token_ids_list = input_ids.tolist()

for pos in range(n_vertices):
    row = {
        "position": pos,
        "token_id": token_ids_list[pos],
        "decoded_piece": tokens_df.loc[pos, "decoded_piece"],
        "frequency": tokens_df.loc[pos, "frequency"],
    }

    for stage in neighbors_by_stage:
        row[stage] = same_token_neighbor_fraction(
            pos,
            neighbors_by_stage[stage],
            token_ids_list
        )

    dominance_rows.append(row)

dominance_df = pd.DataFrame(dominance_rows)
dominance_df.head()

In [ ]:
dominance_summary = (
    dominance_df[["lexical", "initial", "middle", "final"]]
    .agg(["mean", "median"])
    .T
)

dominance_summary

In [ ]:
dominance_summary.plot(
    kind="bar",
    figsize=(8, 5)
)
plt.ylabel("Fração de vizinhos com mesmo token_id")
plt.title("Dominância lexical ao longo das camadas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 16. Vizinhos de ocorrências individuais

In [ ]:
def neighbor_table(position):
    rows = []

    for stage in ["lexical", "initial", "middle", "final"]:
        for rank, nbr in enumerate(neighbors_by_stage[stage][position], start=1):
            rows.append({
                "source_position": position,
                "source_token": tokens_df.loc[position, "decoded_piece"],
                "stage": stage,
                "rank": rank,
                "neighbor_position": nbr,
                "neighbor_token": tokens_df.loc[nbr, "decoded_piece"],
                "same_token_id": (
                    tokens_df.loc[position, "token_id"]
                    == tokens_df.loc[nbr, "token_id"]
                ),
                "similarity": similarity_matrices[stage][position, nbr],
                "neighbor_context": tokens_df.loc[nbr, "context"],
            })

    return pd.DataFrame(rows)


# Exemplo: primeira ocorrência repetida encontrada
repeated_positions = (
    tokens_df[tokens_df["frequency"] > 1]["position"]
    .tolist()
)

if repeated_positions:
    example_position = repeated_positions[0]
    print("Posição escolhida:", example_position)
    print("Contexto:", tokens_df.loc[example_position, "context"])
    display(neighbor_table(example_position))
else:
    print("Nenhum token_id repetido encontrado.")

## 17. Seleção automática de ocorrências repetidas interessantes

In [ ]:
# Seleciona tokens repetidos cuja distância final entre ocorrências seja alta.
interesting_pairs = contextual_distances_df.copy()

if not interesting_pairs.empty:
    interesting_pairs = interesting_pairs.merge(
        tokens_df[["position", "decoded_piece", "context"]],
        left_on="position_a",
        right_on="position",
        how="left"
    ).rename(columns={
        "decoded_piece": "token_a",
        "context": "context_a"
    }).drop(columns=["position"])

    interesting_pairs = interesting_pairs.merge(
        tokens_df[["position", "decoded_piece", "context"]],
        left_on="position_b",
        right_on="position",
        how="left"
    ).rename(columns={
        "decoded_piece": "token_b",
        "context": "context_b"
    }).drop(columns=["position"])

display(interesting_pairs.head(10))

In [ ]:
# Mostra os vizinhos dos dois lados dos três pares mais "separados" no final.
for _, row in interesting_pairs.head(3).iterrows():
    pa = int(row["position_a"])
    pb = int(row["position_b"])

    print("=" * 90)
    print(
        f"token_id={int(row['token_id'])} | "
        f"posição A={pa} | posição B={pb} | "
        f"distância final={row['final_l2_distance']:.4f}"
    )
    print("\nContexto A:", tokens_df.loc[pa, "context"])
    print("Contexto B:", tokens_df.loc[pb, "context"])

    display(
        pd.concat(
            [
                neighbor_table(pa).assign(which="A"),
                neighbor_table(pb).assign(which="B")
            ],
            ignore_index=True
        )
    )

## 18. Detecção de comunidades

In [ ]:
def detect_communities(G):
    if HAS_LOUVAIN:
        partition = community_louvain.best_partition(
            G,
            weight="weight",
            random_state=SEED
        )
        modularity = community_louvain.modularity(
            partition,
            G,
            weight="weight"
        )
        return partition, modularity, "louvain"

    # Fallback para greedy modularity do NetworkX.
    communities = list(
        nx.algorithms.community.greedy_modularity_communities(
            G,
            weight="weight"
        )
    )

    partition = {}
    for cid, community_nodes in enumerate(communities):
        for node in community_nodes:
            partition[node] = cid

    modularity = nx.algorithms.community.modularity(
        G,
        communities,
        weight="weight"
    )

    return partition, modularity, "networkx_greedy_modularity"


community_rows = []
partitions = {}

for stage, G in graphs.items():
    partition, modularity, method = detect_communities(G)
    partitions[stage] = partition

    sizes = Counter(partition.values())

    community_rows.append({
        "stage": stage,
        "method": method,
        "n_communities": len(sizes),
        "modularity": modularity,
        "largest_community": max(sizes.values()),
        "community_sizes": dict(sorted(sizes.items())),
    })

communities_df = pd.DataFrame(community_rows).set_index("stage")
communities_df

## 19. Visualização das comunidades

In [ ]:
def plot_communities(stage):
    G = graphs[stage]
    partition = partitions[stage]

    community_ids = [partition[n] for n in G.nodes()]

    plt.figure(figsize=(10, 8))

    nx.draw_networkx_edges(
        G,
        fixed_pos,
        alpha=0.18,
        width=0.8
    )

    nx.draw_networkx_nodes(
        G,
        fixed_pos,
        node_color=community_ids,
        cmap=plt.cm.tab20,
        node_size=55,
        alpha=0.9
    )

    plt.title(f"Comunidades — {stage}")
    plt.axis("off")
    plt.show()


for stage in ["lexical", "initial", "middle", "final"]:
    plot_communities(stage)

## 20. Tabela consolidada por ocorrência

In [ ]:
occurrence_summary = (
    tokens_df[
        ["position", "token_id", "token", "decoded_piece", "frequency", "context"]
    ]
    .merge(
        jaccard_df,
        on="position",
        how="left"
    )
    .merge(
        dominance_df[
            ["position", "lexical", "initial", "middle", "final"]
        ].rename(columns={
            "lexical": "lexical_dominance",
            "initial": "initial_dominance",
            "middle": "middle_dominance",
            "final": "final_dominance",
        }),
        on="position",
        how="left"
    )
)

for stage in partitions:
    occurrence_summary[f"community_{stage}"] = (
        occurrence_summary["position"]
        .map(partitions[stage])
    )

display(occurrence_summary.head(30))

## 21. Síntese automática dos resultados

In [ ]:
n_occurrences = len(tokens_df)
n_unique_types = tokens_df["token_id"].nunique()
n_repeated_types = int((tokens_df.groupby("token_id").size() > 1).sum())
n_occurrences_from_repeated_types = int(
    tokens_df[tokens_df["frequency"] > 1].shape[0]
)

mean_dominance = dominance_df[
    ["lexical", "initial", "middle", "final"]
].mean()

mean_jaccard_lex_final = jaccard_df["lexical_to_final"].mean()
median_jaccard_lex_final = jaccard_df["lexical_to_final"].median()

print("1. Quantas ocorrências existem?")
print("  ", n_occurrences)

print("\n2. Quantos token_ids distintos?")
print("  ", n_unique_types)

print("\n3. Quantos token_ids aparecem mais de uma vez?")
print("  ", n_repeated_types)

print("\n   Ocorrências pertencentes a token_ids repetidos:")
print("  ", n_occurrences_from_repeated_types)

print("\n4–5. Dominância lexical média por estágio:")
display(mean_dominance.to_frame("mean_same_token_neighbor_fraction"))

print("\n6. Jaccard lexical → final:")
print("   média:", mean_jaccard_lex_final)
print("   mediana:", median_jaccard_lex_final)

print("\n7. Pares repetidos com maior distância contextual final:")
display(contextual_distances_df.head(10))

print("\n8. Comunidades:")
display(communities_df[["n_communities", "modularity", "largest_community"]])

## 22. Sanity checks finais

In [ ]:
# Mesmo conjunto de vértices em todas as redes.
expected_nodes = set(range(n_vertices))

for stage, G in graphs.items():
    assert set(G.nodes()) == expected_nodes
    assert nx.number_of_selfloops(G) == 0

# Cada lista k-NN direcionada deve possuir exatamente K vizinhos,
# desde que existam pelo menos K outros vértices.
expected_k = min(K, n_vertices - 1)

for stage, nbrs in neighbors_by_stage.items():
    lengths = {len(v) for v in nbrs.values()}
    assert lengths == {expected_k}, (stage, lengths)

# Diagonal da similaridade deve ser ~1.
for stage, sim in similarity_matrices.items():
    assert np.allclose(np.diag(sim), 1.0, atol=1e-4), stage

# Embeddings lexicais de mesmo token_id devem ser iguais.
assert same_token_pairs_checked == same_token_lexical_equal

print("Todos os sanity checks passaram.")

## 23. Próximos passos antes de escalar

Após executar o piloto, examine principalmente:

- se a rede lexical é fortemente dominada por ocorrências do mesmo `token_id`;
- se essa dominância cai nas camadas contextuais;
- se o Jaccard lexical → final é baixo o suficiente para indicar reorganização;
- se ocorrências repetidas inicialmente idênticas passam a ter vizinhanças diferentes;
- se a escolha de `k=5` parece adequada;
- se os empates da rede lexical produzem artefatos importantes;
- se a análise deveria excluir pontuação, espaços ou tokens funcionais;
- se palavras de interesse são divididas em subtokens;
- se será necessário controlar frequência por tipo antes de escalar para ~15 mil ocorrências.

### Robustez sugerida para a próxima versão

Depois de validar o pipeline:

1. testar `k ∈ {3, 5, 10}`;
2. testar um corpus real maior;
3. controlar frequência por token_id;
4. selecionar apenas tokens de conteúdo, se necessário;
5. aumentar o número de camadas analisadas;
6. comparar redes com medidas adicionais de persistência;
7. escalar para milhares de ocorrências usando FAISS ou busca aproximada.

## 24. Exportação opcional dos resultados

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("pilot_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

tokens_df.to_csv(OUTPUT_DIR / "tokens.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "network_metrics.csv")
jaccard_df.to_csv(OUTPUT_DIR / "jaccard_by_occurrence.csv", index=False)
dominance_df.to_csv(OUTPUT_DIR / "lexical_dominance.csv", index=False)
communities_df.to_csv(OUTPUT_DIR / "communities_summary.csv")
occurrence_summary.to_csv(OUTPUT_DIR / "occurrence_summary.csv", index=False)

neighbor_rows = []
for stage, nbrs in neighbors_by_stage.items():
    for source, neighbors in nbrs.items():
        for rank, target in enumerate(neighbors, start=1):
            neighbor_rows.append({
                "stage": stage,
                "source": source,
                "rank": rank,
                "target": target,
                "similarity": similarity_matrices[stage][source, target],
            })

neighbors_df = pd.DataFrame(neighbor_rows)
neighbors_df.to_csv(OUTPUT_DIR / "neighbors.csv", index=False)

print("Resultados exportados para:", OUTPUT_DIR.resolve())